# 13 OpenOximetry Logistic Regression Explainability

This notebook adds explainability evidence for the grouped OpenOximetry Logistic Regression baseline. It reports global feature influence from model coefficients and local per-window explanations from standardized feature contributions. These explanations support the research-plan rubric: compactness, consistency with model behavior, and practical usefulness.

**Outputs:**
- `outputs/openox_logreg_global_explainability.csv` — coefficient ranking across grouped folds
- `outputs/openox_logreg_local_explanations.csv` — per-window contributions for all held-out windows
- `outputs/openox_logreg_explanation_rubric_summary.csv` — rubric scores for a balanced 20-window sample

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

## 1. Feature and Label Contract

Load the canonical OpenOximetry feature export. The same 14 engineered PPG features, label, and recording_id used in notebook 11 are used here. SpO2 is used only as label evidence, never as a predictive feature.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUTS = PROJECT_ROOT / "outputs"
FEATURE_PATH = OUTPUTS / "openox_canonical_window_features.csv"

FEATURE_COLUMNS = [
    "mean", "std", "minimum", "maximum", "value_range",
    "rms", "skewness", "kurtosis", "energy", "derivative_std",
    "dominant_frequency_hz", "bandpower_0_5_2_hz",
    "bandpower_2_5_hz", "bandpower_ratio_0_5_2_to_2_5",
]

FEATURE_LABELS = {
    "mean": "mean amplitude",
    "std": "amplitude variability",
    "minimum": "minimum amplitude",
    "maximum": "maximum amplitude",
    "value_range": "amplitude range",
    "rms": "signal energy (RMS)",
    "skewness": "signal skewness",
    "kurtosis": "signal kurtosis",
    "energy": "spectral energy",
    "derivative_std": "derivative variability",
    "dominant_frequency_hz": "dominant frequency",
    "bandpower_0_5_2_hz": "low bandpower (0.5-2 Hz)",
    "bandpower_2_5_hz": "high bandpower (2-5 Hz)",
    "bandpower_ratio_0_5_2_to_2_5": "bandpower ratio (low/high)",
}

df = pd.read_csv(FEATURE_PATH)

for column in FEATURE_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="raise")

df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
df["recording_id"] = df["recording_id"].astype(str)

X = df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
y = df["label"].to_numpy(dtype=int)
groups = df["recording_id"].to_numpy()

print(f"Rows        : {len(df):,}")
print(f"Features    : {len(FEATURE_COLUMNS)}")
print(f"Positive    : {y.sum():,} ({y.mean():.1%})")
print(f"Groups      : {len(np.unique(groups)):,}")

## 2. Grouped Logistic Regression Rebuild

Rebuild the grouped Logistic Regression baseline using `GroupKFold(n_splits=5)` with recording-level grouping. The scaler is fit only on training groups to prevent leakage.

In [ ]:
splitter = GroupKFold(n_splits=5)
splits = list(splitter.split(X, y, groups))

fold_rows = []
local_rows = []
global_rows = []

for fold_index, (train_idx, test_idx) in enumerate(splits, start=1):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="liblinear",
        random_state=42,
    )
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()

    fold_rows.append({
        "fold": fold_index,
        "test_rows": len(test_idx),
        "test_groups": len(np.unique(groups[test_idx])),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "accuracy": float((predictions == y_test).mean()),
        "recall": float(tp / (tp + fn)) if (tp + fn) else np.nan,
    })

    coef = model.coef_[0]
    contributions = X_test * coef

    for feature_name, coefficient in zip(FEATURE_COLUMNS, coef):
        global_rows.append({
            "fold": fold_index,
            "feature_name": feature_name,
            "feature_label": FEATURE_LABELS[feature_name],
            "coefficient": float(coefficient),
            "absolute_coefficient": float(abs(coefficient)),
            "direction": (
                "pushes_toward_low_oxygen_risk" if coefficient > 0
                else "pushes_toward_normal"
            ),
        })

    for row_position, source_index in enumerate(test_idx):
        row_contrib = contributions[row_position]
        top_indices = np.argsort(np.abs(row_contrib))[::-1][:3]

        if y_test[row_position] == 1 and predictions[row_position] == 1:
            quadrant = "TP"
        elif y_test[row_position] == 0 and predictions[row_position] == 1:
            quadrant = "FP"
        elif y_test[row_position] == 0 and predictions[row_position] == 0:
            quadrant = "TN"
        else:
            quadrant = "FN"

        row = {
            "fold": fold_index,
            "source_index": int(source_index),
            "recording_id": groups[source_index],
            "start_time_seconds": float(df.loc[source_index, "start_time_seconds"]),
            "end_time_seconds": float(df.loc[source_index, "end_time_seconds"]),
            "true_label": int(y_test[row_position]),
            "predicted_label": int(predictions[row_position]),
            "probability_low_oxygen_risk": float(probabilities[row_position]),
            "quadrant": quadrant,
        }

        for rank, feature_index in enumerate(top_indices, start=1):
            feature_name = FEATURE_COLUMNS[feature_index]
            standardized_value = X_test[row_position, feature_index]
            contribution = row_contrib[feature_index]

            row[f"top{rank}_feature_name"] = feature_name
            row[f"top{rank}_feature_label"] = FEATURE_LABELS[feature_name]
            row[f"top{rank}_standardized_value"] = float(standardized_value)
            row[f"top{rank}_contribution"] = float(contribution)
            row[f"top{rank}_value_direction"] = (
                "high" if standardized_value > 0 else "low"
            )
            row[f"top{rank}_risk_direction"] = (
                "toward_low_oxygen_risk" if contribution > 0 else "toward_normal"
            )

        local_rows.append(row)

fold_df = pd.DataFrame(fold_rows)
global_fold_df = pd.DataFrame(global_rows)
local_df = pd.DataFrame(local_rows)

print(f"Folds completed  : {len(fold_df)}")
print(f"Global rows      : {len(global_fold_df):,}")
print(f"Local rows       : {len(local_df):,}")
fold_df

## 3. Global Explainability

Aggregate coefficients across grouped folds. Features with larger absolute coefficients and consistent sign across folds are the most influential predictors of oxygen-risk state.

In [ ]:
global_summary_df = (
    global_fold_df
    .groupby(["feature_name", "feature_label"], as_index=False)
    .agg(
        mean_coefficient=("coefficient", "mean"),
        std_coefficient=("coefficient", "std"),
        mean_absolute_coefficient=("absolute_coefficient", "mean"),
        positive_sign_fraction=(
            "coefficient", lambda values: float(np.mean(values > 0))
        ),
        negative_sign_fraction=(
            "coefficient", lambda values: float(np.mean(values < 0))
        ),
        fold_count=("fold", "nunique"),
    )
)

global_summary_df["sign_consistency_rate"] = global_summary_df[
    ["positive_sign_fraction", "negative_sign_fraction"]
].max(axis=1)

global_summary_df["dominant_direction"] = np.where(
    global_summary_df["mean_coefficient"] > 0,
    "pushes_toward_low_oxygen_risk",
    "pushes_toward_normal",
)

global_summary_df = global_summary_df.sort_values(
    ["mean_absolute_coefficient", "sign_consistency_rate"],
    ascending=[False, False],
).reset_index(drop=True)

global_summary_df.head(len(FEATURE_COLUMNS))

## 4. Local Per-Window Explainability

Generate compact natural-language explanations from the top-3 feature contributions for every held-out window. The format is:

> [Risk state] flagged (prob = [probability]). Main contributors: [direction] [feature] ([contribution], [risk direction]); ...

In [ ]:
def format_state(predicted_label: int) -> str:
    return "Low-oxygen risk" if predicted_label == 1 else "Normal oxygen"


def generate_explanation(row: pd.Series) -> str:
    state = format_state(int(row["predicted_label"]))
    prob = float(row["probability_low_oxygen_risk"])

    parts = []
    for rank in [1, 2, 3]:
        name = row[f"top{rank}_feature_label"]
        contrib = float(row[f"top{rank}_contribution"])
        direction = row[f"top{rank}_value_direction"]
        risk = row[f"top{rank}_risk_direction"].replace("_", " ")
        parts.append(f"{direction} {name} ({contrib:+.2f}, {risk})")

    return f"{state} flagged (prob = {prob:.2f}). Main contributors: {parts[0]}; {parts[1]}; {parts[2]}."


local_df["explanation"] = local_df.apply(generate_explanation, axis=1)

display_columns = [
    "quadrant", "true_label", "predicted_label",
    "probability_low_oxygen_risk",
    "top1_feature_label", "top2_feature_label", "explanation",
]

local_df[display_columns].head(10)

## 5. Balanced 20-Window Rubric Sample

Select 5 held-out windows from each confusion-matrix quadrant (TP, FP, TN, FN), evenly spaced by prediction probability. This ensures the rubric assessment covers all model output states.

In [ ]:
sampled_rows = []

for quadrant in ["TP", "FP", "TN", "FN"]:
    quadrant_df = local_df[local_df["quadrant"] == quadrant].copy()

    if quadrant_df.empty:
        continue

    if quadrant in ["TP", "FP"]:
        quadrant_df = quadrant_df.sort_values(
            "probability_low_oxygen_risk", ascending=False
        )
    elif quadrant == "TN":
        quadrant_df = quadrant_df.sort_values(
            "probability_low_oxygen_risk", ascending=True
        )
    else:
        quadrant_df = quadrant_df.sort_values(
            "probability_low_oxygen_risk", ascending=False
        )

    sample_count = min(5, len(quadrant_df))

    if sample_count == 1:
        selected_positions = [0]
    else:
        selected_positions = [
            int(round(pos * (len(quadrant_df) - 1) / (sample_count - 1)))
            for pos in range(sample_count)
        ]

    sampled_rows.extend(quadrant_df.iloc[selected_positions].to_dict("records"))

rubric_sample_df = pd.DataFrame(sampled_rows)

rubric_sample_df["window_id"] = ""
for quadrant in ["TP", "FP", "TN", "FN"]:
    mask = rubric_sample_df["quadrant"] == quadrant
    for count, index in enumerate(rubric_sample_df[mask].index, start=1):
        rubric_sample_df.loc[index, "window_id"] = f"{quadrant}-{count:02d}"

print(f"Selected {len(rubric_sample_df)} windows")

rubric_sample_df[
    ["window_id", "quadrant", "predicted_label",
     "probability_low_oxygen_risk", "explanation"]
].reset_index(drop=True)

## 6. Explanation Clarity Rubric

Score each explanation on three criteria from the research plan (Table 11):

| Criterion | 0 | 1 | 2 |
|---|---|---|---|
| Compactness | Too long for compact display | Borderline but usable | Concise and display-ready |
| Consistency | Conflicts with model behavior | Partially aligned | Clearly aligned |
| Usefulness | No interpretable cue | Limited cue | Clear cue (confidence, feature, direction) |

Pass condition: mean >= 4/6, no criterion scores 0.

In [ ]:
def score_compactness(text: str) -> int:
    word_count = len(text.split())
    if word_count <= 35:
        return 2
    if word_count <= 50:
        return 1
    return 0


def score_consistency(row: pd.Series) -> int:
    predicted = int(row["predicted_label"])
    top1_contrib = float(row["top1_contribution"])
    top2_contrib = float(row["top2_contribution"])

    positive_support = max(top1_contrib, top2_contrib)
    negative_support = min(top1_contrib, top2_contrib)

    if predicted == 1 and positive_support > 0:
        return 2
    if predicted == 0 and negative_support < 0:
        return 2
    if predicted == 1 and positive_support <= 0:
        return 1
    if predicted == 0 and negative_support >= 0:
        return 1
    return 0


def score_usefulness(row: pd.Series) -> int:
    has_prob = pd.notna(row["probability_low_oxygen_risk"])
    has_features = bool(row["top1_feature_label"]) and bool(row["top2_feature_label"])
    has_direction = row["top1_value_direction"] in {"high", "low"}

    if has_prob and has_features and has_direction:
        return 2
    if has_prob and has_features:
        return 1
    return 0


rubric_sample_df["compactness"] = rubric_sample_df["explanation"].apply(
    score_compactness
)
rubric_sample_df["consistency"] = rubric_sample_df.apply(
    score_consistency, axis=1
)
rubric_sample_df["usefulness"] = rubric_sample_df.apply(
    score_usefulness, axis=1
)
rubric_sample_df["total_score"] = (
    rubric_sample_df["compactness"]
    + rubric_sample_df["consistency"]
    + rubric_sample_df["usefulness"]
)

rubric_summary_df = pd.DataFrame([{
    "sample_count": int(len(rubric_sample_df)),
    "mean_compactness": float(rubric_sample_df["compactness"].mean()),
    "mean_consistency": float(rubric_sample_df["consistency"].mean()),
    "mean_usefulness": float(rubric_sample_df["usefulness"].mean()),
    "mean_total_score": float(rubric_sample_df["total_score"].mean()),
    "minimum_compactness": int(rubric_sample_df["compactness"].min()),
    "minimum_consistency": int(rubric_sample_df["consistency"].min()),
    "minimum_usefulness": int(rubric_sample_df["usefulness"].min()),
    "passes_research_plan_gate": bool(
        rubric_sample_df["total_score"].mean() >= 4
        and rubric_sample_df["compactness"].min() > 0
        and rubric_sample_df["consistency"].min() > 0
        and rubric_sample_df["usefulness"].min() > 0
    ),
}])

rubric_summary_df

### Sample Scoring Table

Per-window rubric scores with explanation text.

In [ ]:
rubric_sample_df[
    ["window_id", "quadrant", "true_label", "predicted_label",
     "probability_low_oxygen_risk", "compactness", "consistency",
     "usefulness", "total_score", "explanation"]
].reset_index(drop=True)

## 7. Export Evidence

Write all explainability outputs to `outputs/`.

In [ ]:
global_output_path = OUTPUTS / "openox_logreg_global_explainability.csv"
global_summary_df.to_csv(global_output_path, index=False)
print(f"Wrote: {global_output_path}")

local_output_path = OUTPUTS / "openox_logreg_local_explanations.csv"
local_df.to_csv(local_output_path, index=False)
print(f"Wrote: {local_output_path}")

rubric_output_path = OUTPUTS / "openox_logreg_explanation_rubric_summary.csv"
rubric_summary_df.to_csv(rubric_output_path, index=False)
print(f"Wrote: {rubric_output_path}")

rubric_sample_path = OUTPUTS / "openox_logreg_explanation_rubric_sample.csv"
rubric_sample_df.to_csv(rubric_sample_path, index=False)
print(f"Wrote: {rubric_sample_path}")

## Interpretation Notes

The Logistic Regression baseline is explainable because each standardized PPG feature has a learned coefficient. Global explainability is reported by ranking mean absolute coefficients across grouped folds. Local explainability is reported by multiplying each standardized feature value by its coefficient for a held-out window.

**Positive local contributions** push the model toward the low-oxygen-risk class. **Negative local contributions** push the model toward the normal class. These explanations describe the model's decision logic, not a confirmed medical cause of oxygen desaturation.

The explanation output is suitable for the current research milestone because it is compact, directly tied to model behavior, and compatible with later TinyML deployment. The same coefficient, scaler, threshold, and feature-order logic can later be exported to Arduino C/C++ without requiring a large post-hoc XAI library.